In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import pandas as pd

# 1. Tentukan path folder dan nama file
path_file = '/content/drive/MyDrive/Project UTS NLP/'
nama_file = 'tokopedia_reviews.xlsx'

# 2. Baca file Excel
df = pd.read_excel(path_file + nama_file)

# 3. SESUAIKAN KOLOM (Sudah diperbaiki sesuai dataset Tokopedia kamu)
# Kita ambil 'review_coment' untuk ulasan dan 'star_rating' untuk bintangnya
df_final = df[['review_coment', 'star_rating']].copy()

# 4. Samakan nama kolom agar sinkron dengan kode preprocessing kita sebelumnya
df_final.columns = ['content', 'score']

# 5. Hapus data kosong (NaN) pada ulasan agar tidak error nanti
df_final = df_final.dropna(subset=['content'])

# 6. PENTING: Ambil 10.000 data saja secara acak dari 60.000 data
# Ini supaya proses Preprocessing (Stemming) selesai dalam waktu ~10 menit.
df_final = df_final.sample(n=10000, random_state=42).reset_index(drop=True)

print(f"Berhasil memuat dan menyesuaikan {len(df_final)} data Tokopedia.")
print(df_final.head())

Berhasil memuat dan menyesuaikan 10000 data Tokopedia.
                                             content  score
0  Terima kasih telah berbelanja di Durex Officia...      5
1  Terima kasih telah berbelanja di Dettol Offici...      5
2  Terima kasih telah berbelanja di Dettol Offici...      5
3  Terima kasih telah berbelanja di Dettol &amp; ...      5
4  Terima kasih telah berbelanja di Dettol Offici...      5


In [20]:
import pandas as pd
import re
import pickle
import numpy as np
from sklearn.model_selection import train_test_split
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from nltk.tokenize import word_tokenize

# Load Data
df = pd.read_excel('/content/drive/MyDrive/Project UTS NLP/tokopedia_reviews.xlsx')
df = df[['review_coment', 'star_rating']].dropna().sample(10000, random_state=42)
df.columns = ['content', 'label']

# Labeling: Bintang 4-5 (Positif/1), Bintang 1-3 (Negatif/0)
df['label'] = df['label'].apply(lambda x: 1 if x >= 4 else 0)

# Preprocessing
stop_factory = StopWordRemoverFactory()
stopword_remover = stop_factory.create_stop_word_remover()
stemmer = StemmerFactory().create_stemmer()

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'[^\w\s]', '', text) # Perbaikan RegEx
    text = stopword_remover.remove(text)
    text = stemmer.stem(text)
    return text

df['content_cleaned'] = df['content'].apply(clean_text)
df['tokens'] = df['content_cleaned'].apply(word_tokenize)

In [22]:
# Jalankan ini dulu agar kolom 'tokenized' tercipta
from nltk.tokenize import word_tokenize

# Pastikan content_cleaned sudah ada
df['tokenized'] = df['content_cleaned'].apply(word_tokenize)

# Cek apakah kolom sudah muncul
print(df.columns)

Index(['content', 'label', 'content_cleaned', 'tokens', 'tokenized'], dtype='object')


In [23]:
# A. TF-IDF
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf_vec = TfidfVectorizer(max_features=5000)
X_tfidf = tfidf_vec.fit_transform(df['content_cleaned'])

# B. Word Embedding (Word2Vec)
from gensim.models import Word2Vec
w2v_model = Word2Vec(sentences=df['tokenized'], vector_size=100, window=5, min_count=1, workers=4)

def get_vector(tokens):
    vecs = [w2v_model.wv[word] for word in tokens if word in w2v_model.wv]
    return np.mean(vecs, axis=0) if vecs else np.zeros(100)

X_w2v = np.array([get_vector(t) for t in df['tokenized']])

In [25]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score

# Split Data
X_train, X_test, y_train, y_test = train_test_split(X_tfidf, df['label'], test_size=0.2, random_state=42)

models = {
    "Naive Bayes": MultinomialNB(),
    "Logistic Regression": LogisticRegression(),
    "SVM": SVC(probability=True, kernel='linear')
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    results[name] = acc
    print(f"\n=== {name} ===")
    print(classification_report(y_test, y_pred))

path = '/content/drive/MyDrive/Project UTS NLP/'

# Simpan Model Terbaik (Misal SVM) dan Vectorizer
with open(path + 'model_sentiment_tokopedia.pkl', 'wb') as f:
    pickle.dump(models["SVM"], f)
with open(path + 'tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(tfidf_vec, f)


=== Naive Bayes ===
              precision    recall  f1-score   support

           0       0.81      0.95      0.88        60
           1       1.00      0.99      1.00      1940

    accuracy                           0.99      2000
   macro avg       0.91      0.97      0.94      2000
weighted avg       0.99      0.99      0.99      2000


=== Logistic Regression ===
              precision    recall  f1-score   support

           0       0.93      0.93      0.93        60
           1       1.00      1.00      1.00      1940

    accuracy                           1.00      2000
   macro avg       0.97      0.97      0.97      2000
weighted avg       1.00      1.00      1.00      2000


=== SVM ===
              precision    recall  f1-score   support

           0       0.93      0.95      0.94        60
           1       1.00      1.00      1.00      1940

    accuracy                           1.00      2000
   macro avg       0.97      0.97      0.97      2000
weighted av

In [26]:
# Uji LogReg pada Word2Vec
X_train_w, X_test_w, y_train_w, y_test_w = train_test_split(X_w2v, df['label'], test_size=0.2, random_state=42)
lr_w2v = LogisticRegression().fit(X_train_w, y_train_w)
acc_w2v = accuracy_score(y_test_w, lr_w2v.predict(X_test_w))

print(f"\nPerbandingan Ekstraksi Fitur (Logistic Regression):")
print(f"TF-IDF Accuracy: {results['Logistic Regression']:.4f}")
print(f"Word2Vec Accuracy: {acc_w2v:.4f}")


Perbandingan Ekstraksi Fitur (Logistic Regression):
TF-IDF Accuracy: 0.9960
Word2Vec Accuracy: 0.9960


In [27]:
import gradio as gr

def predict_sentiment(text):
    cleaned = clean_text(text)
    if not cleaned: return "Teks kosong", 0

    # Load & Predict
    vec = tfidf_vec.transform([cleaned])
    res = models["SVM"].predict(vec)[0]

    label = "😊 Positif" if res == 1 else "😞 Negatif"
    return label

demo = gr.Interface(
    fn=predict_sentiment,
    inputs=gr.Textbox(label="Masukkan Review"),
    outputs=gr.Label(label="Hasil Prediksi"),
    title="Deployment Project DM206",
    theme="glass"
)
demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://b69ea35fabbce7f1bb.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
